<a href="https://colab.research.google.com/github/francoissouza/Winding-MMF-optimization/blob/main/Winding_MMF_optimization_Francois_de_S_Martins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab에서 열기"/></a>

<p align = "justify">

# MMF 왜곡 최소화를 통한 권선 분포 최적화


---


## 초록
본 논문에서는 단상 또는 다상 시스템에서 낮은 기자력(MMF) 왜곡을 보이는 전기기계의 최적의 권선 방식을 찾기 위해 유전 알고리즘을 사용합니다.

## 서론
먼저, 권선 분포표(WDT)와 페이저 스타 기법을 사용하여 실현 가능하고 대칭적인 전기기계 권선을 생성하는 전략을 연구합니다. 층수, 슬롯수, 극수, 코일 피치(이중층 다이아몬드/임브리케이트 권선의 경우), 그리고 빈 슬롯을 변화시키며 단상 및 다상 기계를 생성하고 분석하는 알고리즘을 구현합니다. [Pyrhoenen][Smolensky][Casuso].

그 다음, 공극 기계에서의 기자력(MMF)을 조사합니다. 관심사는 슬롯에 있는 각 도체가 생성하는 각 MMF의 계산적 합성을 사용하여 권선의 MMF 곡선에서 왜곡을 결정하는 것입니다. 푸리에 변환을 사용하여 MMF를 통한 고조파 왜곡의 계수를 결정합니다. [4][5]

마지막으로, 최적의 권선 세트, 즉 MMF 왜곡이 적은 권선을 찾기 위해 유전 알고리즘을 사용합니다.

## 대칭 권선 설계 조건
권선이 실현 가능한 것으로 간주되려면 상들을 슬롯을 따라 분포시킬 수 있고 모든 상에 의한 균일한 점유를 달성할 수 있어야 합니다 [Pyrhoenen]. 실현 가능하려면, 권선은 두 가지 조건을 만족해야 합니다:

* **첫 번째 조건**: 권선은 식 (1)에서 보인 바와 같이 상당 정수 개의 코일을 가져야 합니다.

$$n_{\text{lay}}\frac{N - n_{\text{es}}}{2m} \in \mathbb{N} \qquad (1)$$

  여기서 $N$은 슬롯 수, $n_{\text{es}}$는 빈 슬롯 수(즉, 코일 측면이 없는 슬롯), $n_{\text{lay}}$는 층 수, $m$은 상의 수입니다.

* **두 번째 조건**: 상 권선 간의 각도 $\alpha_{\text{ph}}$는 슬롯 간 각도 $\alpha_{z}$의 정수배여야 합니다.
  * 정상 시스템의 경우(상의 수 $m$이 홀수이거나 3의 배수), 이는 식 (2)로 표현됩니다

$$\frac{\alpha_{\text{ph}}}{\alpha_{z}} = \frac{2\pi N}{m \cdot 2\pi \cdot t} = \frac{N}{mt} \in \mathbb{N} \qquad (2)$$

  * 축소 시스템의 경우(상의 수 $m$이 짝수), 이는 식 (3)으로 표현됩니다

$$\frac{\alpha_{\text{ph}}}{\alpha_{z}} = \frac{\pi N}{m \cdot 2\pi \cdot t} = \frac{N}{2mt} \in \mathbb{N} \qquad (3)$$

  여기서 $t$는 $N$과 $p$의 최대공약수이고, $p$는 권선의 극쌍 수입니다.

두 조건이 모두 만족되면, 실현 가능한 권선이 가능합니다. 그렇지 않으면, 대칭 권선을 만들 수 없습니다.

실현 가능한 권선을 달성한 후에는 대칭 조건을 확인해야 합니다. 이는 페이저 스타를 분석함으로써 수행되는데, 이는 각 상의 모든 코일-측면 기여도를 합산하여 계산되며 전압 또는 기전력(EMF)의 상 다이어그램을 제공합니다. 이 과정은 24개 슬롯, 2개 극쌍, 단일층 권선을 가진 3상 기계를 사용하여 아래에 설명됩니다.

이 EMF 기계 스타는 중첩된 $t=2$개의 EMF 부-스타로 구성되며, 각 부-스타는 $N/t$개의 페이저로 형성되어 서로 다음과 같은 전기적 각도 $\alpha$만큼 위상차를 가집니다.

In [2]:
# !pip install tqdm

import numpy as np
import pandas as pd
import pdb
import math
import cmath
from fractions import Fraction
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm 

# Google Colab 환경이 아닌 경우를 위한 수정
try:
    %load_ext google.colab.data_table
except:
    print("Google Colab 환경이 아닙니다. 일반 pandas 표시를 사용합니다.")

%pdb off

Google Colab 환경이 아닙니다. 일반 pandas 표시를 사용합니다.
Automatic pdb calling has been turned OFF


In [4]:
class winding:
  def __init__(self, m, N, p, n_lay, y, n_es):
    self.error = ""
    self.m = m; self.N = N; self.p = p; self.n_lay = n_lay; self.y = y; self.n_es = n_es
    self.Q =  Fraction(self.N / (2*self.p*self.m)).limit_denominator(1000) * (n_es > 0)
    self.q = Fraction((self.N-self.n_es) / (2 * self.p * self.m)).limit_denominator(1000)
    self.a = math.floor(self.q)
    self.z = int(self.n_lay * self.p * (self.q - self.a))
    self.condition_1 = self.n_lay * (self.N - self.n_es) / (2 * self.m)
    self.t = math.gcd((self.N), self.p)
    self.sys_type = self.compute_type()
    self.condition_2 = ((self.N)/(self.m*self.t))*(self.sys_type == "Normal" or self.sys_type == "Non-reduced") + ((self.N)/(2*self.m*self.t))*(self.sys_type == "Reduced")
    self.zeta = ((self.m-1)/2)*(self.sys_type == "Normal") + ((self.m/2)-1)*(self.sys_type == "Non-reduced") + (0)*(self.sys_type == "Reduced")
    self.alpha_ph = (360/self.m)*(self.sys_type == "Normal" + self.sys_type == "Non-reduced") + (180/self.m)*(self.sys_type == "Reduced")
    self.t_line = math.gcd(self.z, self.p)*(self.n_es <= 0) + (1)*(self.n_es > 0)
    self.N_line = int(self.N/self.t_line)
    self.p_line = int(self.p/self.t_line)
    self.alpha_g = 180/self.m
    self.m_u = self.gpf(self.m) # Number of phases of subsystem groups that compound a reduced winding
    self.m_g = int(self.m/self.m_u) # Number of subsystems that compound a reduced winding
    self.alpha = self.t*360/self.N
    self.WDT = 0
    self.phi_k = np.array([slot*self.alpha for slot in np.arange(self.N * self.n_lay)])
    self.star_of_slots = np.zeros((self.m, 2))
    self.symmetric_star = False
    self.winding_scheme = np.zeros((self.n_lay, self.N))
    self.feasible = self.is_feasible()

    if self.feasible:
      self.star_elements = np.zeros((self.m, int(self.n_lay * (self.N)/self.m), 2))
      self.WDT = self.compute_WDT()
      self.compute_star_of_slots()
    else:
      self.error += str("The winding is not feasible.\n" + "Condition 1:" + str(self.condition_1) + "\nCondition 2:" + str(self.condition_2))

    self.feasible_and_symmetric = self.feasible and self.symmetric_star

  def is_feasible(self): # Verify if the machine is feasible
    if self.n_es >= self.N:
      self.error += " There are more empty slots than the total slots disponible."
      return False
    return (self.condition_1 % 1 == 0) and (self.condition_2 % 1 == 0)

  def compute_type(self): # Machine type: Normal, Non-reduced or Reduced
    if (self.m % 2):
      return "Normal"
    elif not(self.m % 2) and (self.m % 3 == 0):
      return "Non-reduced"
    return "Reduced"

  def gpf(self, n): # Find the largest prime factor
    maxPrime = -1
    while n % 2 == 0:
        maxPrime = 2
        n >>= 1
    for i in range(3, int(math.sqrt(n)) + 1, 2):
        while n % i == 0:
            maxPrime = i
            n = n / i
    if n > 2:
        maxPrime = n  
    return int(maxPrime)

  def print_error(self):
    print(self.error)

  def compute_WDT(self): # Compute the Winding Distribution Table (WDT)
    if self.feasible:
      for k in np.arange(1, self.t_line + 1): # Compute the primitie WDT form
        j = 0
        while (self.N_line + j) % self.m != 0:
          j += 1
        WDT_base = np.zeros(self.N_line + j)
        i = 0
        for slot in np.arange((k-1)*self.N_line + 1, k*self.N_line + 1):
          while WDT_base[i] != 0:
            i = (i + 1)*(i < (self.N_line-1)) + (i-self.N_line)*(i >= (self.N_line-1))
          WDT_base[i] = slot
          i += (self.p_line)
          while i >= self.N_line:
            i = (i)*(i < (self.N_line)) + (i-self.N_line)*(i >= (self.N_line))
        WDT_base = WDT_base.reshape((self.m, int(np.size(WDT_base)/self.m)))
        try:
          WDT_23 = np.hstack((WDT_23, WDT_base[:, : math.ceil((self.N_line-self.n_es)/(2*self.m))]))
          WDT_14 = np.hstack((WDT_14, WDT_base[:, math.ceil((self.N_line-self.n_es)/(2*self.m)) :]))
        except:
          WDT_23 = WDT_base[:, : math.ceil((self.N_line-self.n_es)/(2*self.m))]
          WDT_14 = WDT_base[:, math.ceil((self.N_line-self.n_es)/(2*self.m)) :]
      WDT = np.hstack((WDT_23, WDT_14))
      if self.sys_type == "Normal" or self.sys_type == "Non-reduced":
        if self.n_es > 0:
          WDT = np.hstack((WDT_23, WDT_14))
          WDT_23 = WDT[:, : math.ceil((self.N - self.n_es)/(2*self.m))]
          WDT_14 = WDT[:, math.ceil((self.N - self.n_es)/(2*self.m)) : -math.ceil(self.n_es/self.m)]
        if np.size(WDT_14) != 0:
          WDT_14 = - np.roll(WDT_14, - int(self.zeta*np.size(WDT_14, 1)))
          self.WDT_1lay = np.hstack((WDT_23, WDT_14))
          if self.n_lay == 2:
            WDT_23_2lay = np.where(((abs(WDT_23)+self.y)//self.N > 0) , (abs(WDT_23)+self.y) % self.N, (abs(WDT_23)+self.y))
            WDT_23_2lay = np.where(WDT_23_2lay == 0, self.N, WDT_23_2lay)
            WDT_23_2lay = - np.sign(WDT_23)*WDT_23_2lay
            WDT_14_2lay = np.where(((abs(WDT_14)+self.y)//self.N > 0) , (abs(WDT_14)+self.y) % self.N, (abs(WDT_14)+self.y))
            WDT_14_2lay = np.where(WDT_14_2lay == 0, self.N, WDT_14_2lay)
            WDT_14_2lay = - np.sign(WDT_14)*WDT_14_2lay
            WDT_23 = np.hstack((WDT_23, WDT_23_2lay))
            WDT_14 = np.hstack((WDT_14, WDT_14_2lay))
            self.WDT_2lay = np.hstack((WDT_23_2lay, WDT_14_2lay))
          WDT = np.hstack((WDT_23, WDT_14))
        else:
          if self.n_lay == 2:
            WDT_23_2lay = np.where(((abs(WDT_23)+self.y)//self.N > 0) , (abs(WDT_23)+self.y) % self.N, (abs(WDT_23)+self.y))
            WDT_23_2lay = np.where(WDT_23_2lay == 0, self.N, WDT_23_2lay)
            WDT_23_2lay = - np.sign(WDT_23)*WDT_23_2lay
            WDT_23 = np.hstack((WDT_23, WDT_23_2lay))
            self.WDT_2lay = WDT_23_2lay
          self.WDT_1lay = WDT_23
          WDT = WDT_23
      if self.sys_type == "Reduced":
        if self.n_es > 0:
          WDT = np.hstack((WDT_23, WDT_14))
          WDT_23 = WDT[:, : math.ceil((self.N - self.n_es)/(2*self.m))]
          WDT_14 = WDT[:, math.ceil((self.N - self.n_es)/(2*self.m)) : -math.ceil(self.n_es/self.m)]
        if np.size(WDT_14) != 0:
          if np.size(WDT_23, axis=1) > np.size(WDT_14, axis=1):
            WDT_14 = np.pad(WDT_14, ((0,0),(0,np.size(WDT_23, axis=1) - np.size(WDT_14, axis=1))), mode="constant", constant_values=0.5)
          WDT_23_sup, WDT_23_inf = np.split(WDT_23, 2, 0)
          WDT_14_sup, WDT_14_inf = np.split(WDT_14, 2, 0)
          WDT_23 = np.vstack((WDT_23_sup, WDT_14_sup))
          WDT_14 = np.vstack((- WDT_23_inf, - WDT_14_inf))
          if self.m_u != 2:
            even_phases = 1
            for n in np.arange(self.m_u * self.m_g):
              if not even_phases % 2:
                WDT_23[n] = - WDT_23[n]
                WDT_14[n] = - WDT_14[n]
              even_phases = (even_phases + 1) - (int(even_phases/self.m_u)*self.m_u)
          a = 0
          b = int(self.m/2)
          WDT_23_aux = np.zeros(np.shape(WDT_23))
          WDT_14_aux = np.zeros(np.shape(WDT_14))
          for j in np.arange(1, self.m + 1): # Merge matrix rows
            if j % 2:
              WDT_23_aux[j-1] = WDT_23[a]
              WDT_14_aux[j-1] = WDT_14[a]
              a += 1
            else:
              WDT_23_aux[j-1] = WDT_23[b]
              WDT_14_aux[j-1] = WDT_14[b]
              b += 1
          WDT_23 = WDT_23_aux
          WDT_14 = WDT_14_aux
          self.WDT_1lay = np.hstack((WDT_23, WDT_14))
          if self.n_lay == 2:
            WDT_23_2lay = np.where(((abs(WDT_23)+self.y)//self.N > 0) , (abs(WDT_23)+self.y) % self.N, (abs(WDT_23)+self.y))
            WDT_23_2lay = np.where(WDT_23_2lay == 0, self.N, WDT_23_2lay)
            WDT_23_2lay = - np.sign(WDT_23)*WDT_23_2lay
            WDT_14_2lay = np.where(((abs(WDT_14)+self.y)//self.N > 0) , (abs(WDT_14)+self.y) % self.N, (abs(WDT_14)+self.y))
            WDT_14_2lay = np.where(WDT_14_2lay == 0, self.N, WDT_14_2lay)
            WDT_14_2lay = - np.sign(WDT_14)*WDT_14_2lay
            WDT_23 = np.hstack((WDT_23, WDT_23_2lay))
            WDT_14 = np.hstack((WDT_14, WDT_14_2lay))
            self.WDT_2lay = np.hstack((WDT_23_2lay, WDT_14_2lay))
          WDT = np.hstack((WDT_23, WDT_14))
          WDT = np.where(WDT % 1 != 0, 0, WDT)
        else:
          if self.n_lay == 2:
            WDT_23_2lay = np.where(((abs(WDT_23)+self.y)//self.N > 0) , (abs(WDT_23)+self.y) % self.N, (abs(WDT_23)+self.y))
            WDT_23_2lay = np.where(WDT_23_2lay == 0, self.N, WDT_23_2lay)
            WDT_23_2lay = - np.sign(WDT_23)*WDT_23_2lay
            WDT_23 = np.hstack((WDT_23, WDT_23_2lay))
            self.WDT_2lay = WDT_23_2lay
          self.WDT_1lay = WDT_23
          WDT = WDT_23
    return WDT

  def compute_star_of_slots(self): # Compute the Star of Slots and Star Symmetry
    for ph in np.arange(self.m):
      for i in np.arange(np.size(self.WDT, axis=1)):
        self.star_elements[int(ph), int(i), 0] = np.cos(((np.sign(self.WDT[int(ph), int(i)]) < 0)*180 + self.phi_k[int(abs(self.WDT[int(ph), int(i)])-1)])* np.pi/180)
        self.star_elements[int(ph), int(i), 1] = np.sin(((np.sign(self.WDT[int(ph), int(i)]) < 0)*180 + self.phi_k[int(abs(self.WDT[int(ph), int(i)])-1)])* np.pi/180)
      self.star_of_slots[ph] = np.sum(self.star_elements[ph], axis=0)
    magnitude = np.array([np.linalg.norm(self.star_of_slots[ph]) for ph in np.arange(self.m)])
    comp = np.array([complex(self.star_of_slots[ph,0], self.star_of_slots[ph,1]) for ph in np.arange(self.m)])
    angle = np.angle(comp,deg=True)
    angle = np.round(angle, 4)
    angle.sort()
    target_angle = (360/self.m)*(self.sys_type == "Normal" or self.sys_type == "Non-reduced") + (180/self.m)*(self.sys_type == "Reduced")
    self.symmetric_star = True
    if self.m == 1:
      if (np.size(self.WDT > 0) != np.size(self.WDT < 0)):
        self.symmetric_star = False
        self.error += " This single-phase winding have not coil side conductor symmetry."
    else:
      for d in np.arange(np.size(angle)-1):
        if not(math.isclose(target_angle, np.diff(angle)[d], rel_tol=1e-02, abs_tol=1e-02)) and not(math.isclose(target_angle, 360 - np.diff(angle)[d], rel_tol=1e-02, abs_tol=1e-02)):
          self.symmetric_star = False
          self.error += " This winding have not symmetric EMF (divergence in phasor angles)."
        if not math.isclose(0, np.diff(magnitude)[d], rel_tol=1e-02, abs_tol=1e-02):
          self.symmetric_star = False
          self.error += " This winding have not symmetric EMF (divergence in magnitude of phasors)."
      if np.all(self.star_of_slots <= 1e-02):
        self.symmetric_star = False
        self.error = " No EMF produced because in this winding all phasors are zero"

  def plot_star_and_scheme(self):
    if self.feasible:
      star_elem_to_plot = self.star_elements
      for ph in np.arange(self.m):
        for i in np.arange(1, np.size(self.WDT, axis=1)):
          star_elem_to_plot[int(ph), int(i), 0] = star_elem_to_plot[int(ph), int(i-1), 0] + star_elem_to_plot[int(ph), int(i), 0]
          star_elem_to_plot[int(ph), int(i), 1] = star_elem_to_plot[int(ph), int(i-1), 1] + star_elem_to_plot[int(ph), int(i), 1]
      self.compute_winding_scheme()
      fig1 = go.Figure(layout=go.Layout(title=go.layout.Title(text="Slot Voltage Phasor")))
      fig2 = go.Figure(layout=go.Layout(title=go.layout.Title(text="Conductors arrangement of first layer")))
      fig3 = go.Figure(layout=go.Layout(title=go.layout.Title(text="Conductors arrangement of second layer")))
      if n_lay == 1:
        tabel_colors = [px.colors.qualitative.Plotly[abs(int(e[0]))-1]*(abs(int(e[0]))>0) + "white"*(abs(int(e[0]))<=0) for e in np.transpose(self.winding_scheme)]
      else:
        tabel_colors = [[px.colors.qualitative.Plotly[abs(int(e[0]))-1]*(abs(int(e[0]))>0) + "white"*(abs(int(e[0]))<=0), px.colors.qualitative.Plotly[abs(int(e[1]))-1]*(abs(int(e[1]))>0) + "white"*(abs(int(e[1]))<=0)] for e in np.transpose(self.winding_scheme)]
      fig4 = go.Figure(data=[go.Table(header=dict(values=np.arange(1, self.N+1), align='center'), cells=dict(values=np.transpose(self.winding_scheme), fill_color=tabel_colors, font=dict(color='white')))])
      for ph in np.arange(self.m):
        fig1.add_trace(go.Scatter(x=np.hstack(([0],star_elem_to_plot[ph,:,0])), y=np.hstack(([0],star_elem_to_plot[ph,:,1])), line = dict(dash='dot', color=px.colors.qualitative.Plotly[ph]), name="Coil EMF " + str(ph+1)))
        fig1.add_trace(go.Scatter(x=[0, self.star_of_slots[ph, 0]], y=[0, self.star_of_slots[ph, 1]], line = dict(color=px.colors.qualitative.Plotly[ph]) , name="Phase " + str(ph+1)))
        fig2.add_trace(go.Scatter(mode="markers", marker_symbol=6, marker=dict(size=10, opacity=1, color=px.colors.qualitative.Plotly[ph]), x=abs(self.WDT_1lay[ph,np.where(self.WDT_1lay[ph] < 0)[0]]), y=abs(np.zeros(np.shape(self.WDT_1lay[ph,np.where(self.WDT_1lay[ph] < 0)[0]]))),name="Positive EMF " + str(ph+1)))
        fig2.add_trace(go.Scatter(mode="markers", marker_symbol=5, marker=dict(size=10, opacity=1, color=px.colors.qualitative.Plotly[ph]), x=abs(self.WDT_1lay[ph,np.where(self.WDT_1lay[ph] > 0)[0]]), y=abs(np.zeros(np.shape(self.WDT_1lay[ph,np.where(self.WDT_1lay[ph] > 0)[0]]))), name="Negative EMF " + str(ph+1)))
        try:
          fig3.add_trace(go.Scatter(mode="markers", marker_symbol=6, marker=dict(size=10, opacity=1, color=px.colors.qualitative.Plotly[ph]), x=abs(self.WDT_2lay[ph,np.where(self.WDT_2lay[ph] < 0)[0]]), y=abs(np.zeros(np.shape(self.WDT_2lay[ph,np.where(self.WDT_2lay[ph] < 0)[0]]))),name="Positive EMF " + str(ph+1)))
          fig3.add_trace(go.Scatter(mode="markers", marker_symbol=5, marker=dict(size=10, opacity=1, color=px.colors.qualitative.Plotly[ph]), x=abs(self.WDT_2lay[ph,np.where(self.WDT_2lay[ph] > 0)[0]]), y=abs(np.zeros(np.shape(self.WDT_2lay[ph,np.where(self.WDT_2lay[ph] > 0)[0]]))), name="Negative EMF " + str(ph+1)))
        except:
          pass
        for cond in np.arange((self.N-self.n_es)/self.m):
          fig2.add_vline(x=abs(self.WDT_1lay[ph, int(cond)]), line_color=px.colors.qualitative.Plotly[ph])
          try:
            fig3.add_vline(x=abs(self.WDT_2lay[ph, int(cond)]), line_color=px.colors.qualitative.Plotly[ph])
          except:
            pass
      slots_to_plot = np.arange(1, self.N+1)
      if self.n_lay == 2:
        slots_to_plot = np.hstack((np.arange(1, self.N+1), np.arange(1, self.N+1)))
      cond_to_plot = self.winding_scheme.reshape(1, np.size(self.winding_scheme))
      fig1.update_yaxes(scaleanchor = "x", scaleratio = 1)
      fig1.show()
      fig2.update_yaxes(scaleanchor = "x", scaleratio = 1)
      fig2.update_xaxes(nticks=self.N)
      fig2.update_layout(xaxis_title="Slots", yaxis_title="Conductor current direction of first layer")
      fig2.show()
      if n_lay == 2:
        fig3.update_yaxes(scaleanchor = "x", scaleratio = 1)
        fig3.update_xaxes(nticks=self.N)
        fig3.update_layout(xaxis_title="Slots", yaxis_title="Conductor current direction of second layer")
        fig3.show()
      fig4.update_layout(xaxis_title="Slots", yaxis_title="Winding Scheme")
      fig4.show()

  def compute_winding_scheme(self):
    if self.feasible:
      for ph in np.arange(self.m):
        for i in np.arange(0, (self.N-self.n_es) / self.m):
          self.winding_scheme[0, int(abs(self.WDT_1lay[int(ph), int(i)])-1)] = np.sign(self.WDT_1lay[int(ph), int(i)])*(ph+1)
          if self.n_lay == 2:
            self.winding_scheme[1, int(abs(self.WDT_2lay[int(ph), int(i)])-1)] = np.sign(self.WDT_2lay[int(ph), int(i)])*(ph+1)
    else:
       self.error = " Is not possible compute the winding scheme because this configuration is not feaseble."

In [5]:
# ------------------------------------------------------------------------------
# Try the following code to compute the star-of-slots, the conductior direction 
# disposition for all layers and the table of winding schemes.
# ------------------------------------------------------------------------------

m = 6
N = 36
p = 2
n_lay = 2
y = 7
n_es = 6

wd = winding( m, N, p, n_lay, y, n_es)
print(wd.WDT)
wd.print_error()
wd.plot_star_and_scheme()

[[  1.  19.   2.  -8. -26.  -9. -26.  -9.  33.  16.]
 [  4.  22.   5. -11. -29. -12. -29. -12.  36.  19.]
 [  7.  25.   8. -14. -32. -15. -32. -15.   3.  22.]
 [ 10.  28.  11. -17. -35. -18. -35. -18.   6.  25.]
 [ 13.  31.  14. -20.  -2. -21. -20.  -3.  27.  10.]
 [ 16.  34.  17. -23.  -5. -24. -23.  -6.  30.  13.]]



ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
def feasible_winding_table(m, n_lay, y, n_es, N_min, N_max, p_min, p_max):
  table = np.zeros((N_max,p_max))
  for N in np.arange(N_min, N_max+1):
    for p in np.arange(p_min, p_max+1):
      wd = winding( m, N, p, n_lay, y, n_es)
      table[N-1, p-1] = int(wd.feasible)
  return table

def feasible_and_symmetric_winding_table(m, n_lay, y, n_es, N_min, N_max, p_min, p_max):
  table = np.zeros((N_max,p_max))
  for N in np.arange(N_min, N_max+1):
    for p in np.arange(p_min, p_max+1):
      wd = winding( m, N, p, n_lay, y, n_es)
      table[N-1, p-1] = int(wd.feasible_and_symmetric)
  return table

def plot_winding_table(df):
  df.replace(0, "-", inplace = True)
  df.replace(1, "OK", inplace = True)
  df.insert(0, -1, np.arange(1, N_max+1))
  numer_colors = [[px.colors.qualitative.Plotly[0] if df.iloc[row, col] == "OK" else "#E2E2E2" for row in np.arange(N_max)] for col in np.arange(p_max+1)]
  fig = go.Figure(data=[go.Table(header=dict(values=list(df.columns+1)), header_prefix="p=" ,cells=dict(values=[df.iloc[:, i] for i in np.arange(p_max+1)], fill_color=numer_colors))])
  fig.show()


In [ ]:
# ------------------------------------------------------------------------------
# Robustness test.
# ------------------------------------------------------------------------------

m_max = 12
n_lay_max = 2
y_max = 8
N_max = 96
p_max = 10
n_es_max = 15

print("Robustness test of winding creator algorithm.")
pbar1 = tqdm(range(1, m_max+1))
pbar1.set_description("Phases:")

try:
  for m in pbar1:
    for n_lay in tqdm(range(1, n_lay_max+1), leave=False, ascii=True, desc="Layers"):
      for y in tqdm(range(1, y_max+1), leave=False, ascii=True, desc="Coil pitch"):
        for n_es in tqdm(range(0, n_es_max+1),  leave=False, ascii=True, desc="Empty Slots"):
          df = pd.DataFrame(feasible_and_symmetric_winding_table(m, n_lay, y, n_es, 1, N_max, 1, p_max))
  print("Successful tested.")
except:
  print("Error! The algorithm breaks")

Robustness test of winding creator algorithm.


  0%|          | 0/12 [00:00<?, ?it/s]

Layers:   0%|          | 0/2 [00:00<?, ?it/s]

Coil pitch:   0%|          | 0/8 [00:00<?, ?it/s]

Empty Slots:   0%|          | 0/16 [00:00<?, ?it/s]

...

Empty Slots:   0%|          | 0/16 [00:00<?, ?it/s]

Empty Slots:   0%|          | 0/16 [00:00<?, ?it/s]

Empty Slots:   0%|          | 0/16 [00:00<?, ?it/s]

Empty Slots:   0%|          | 0/16 [00:00<?, ?it/s]

Successful tested.


In [3]:
# ------------------------------------------------------------------------------
# Try the following code to compute the feasible winding table.
# ------------------------------------------------------------------------------

m = 4; n_lay = 2; y = 3; n_es = 0
N_min = 1; N_max = 40  
p_min = 1; p_max = 20

df = pd.DataFrame(feasible_winding_table(m, n_lay, y, n_es, N_min, N_max, p_min, p_max))
plot_winding_table(df)

NameError: name 'feasible_winding_table' is not defined

In [ ]:
# ------------------------------------------------------------------------------
# Try the following code to compute the feasible and symmetric winding table.
# ------------------------------------------------------------------------------

m = 4; n_lay = 2; y = 3; n_es = 0
N_min = 1; N_max = 40  
p_min = 1; p_max = 20

df = pd.DataFrame(feasible_and_symmetric_winding_table(m, n_lay, y, n_es, N_min, N_max, p_min, p_max))
plot_winding_table(df)